In [1]:
import sys
import os
# sys.path.append('..')
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, PROJECT_ROOT)

import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torch.utils.tensorboard import SummaryWriter
import numpy as np
import pandas as pd


print(os.getcwd())
import datetime
from src.models.cmae import CMAE
from src.data.load_cifar10 import get_cifar10_loaders
from src.data.load_cifar100 import get_cifar100_loaders, create_and_load_subset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)



C:\Users\aczar\Desktop\polibuda\ML_projekt\src\notebooks_test_train
cuda


In [2]:
# Dane
# train_loader, val_loader, test_loader = get_cifar10_loaders(batch_size=64)
# train_loader, val_loader, test_loader = get_cifar100_loaders(batch_size=64)
_, selected_classes, train_loader, val_loader, test_loader = create_and_load_subset(
    num_classes=5,
    batch_size=64
)
print(f"Trenowanie na klasach: {selected_classes}")


Wylosowano nowe klasy: [45, 13, 47, 22, 69]
Trenowanie na klasach: [45, 13, 47, 22, 69]


In [3]:
# Model
model = CMAE(latent_dim=256).to(device)

# optimizer
# optimizer = optim.Adam(model.parameters(), lr=0.001)


optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.05) # AdamW niby lepszy dla contrastive learning

In [4]:
# Trening

BASE_DIR = os.getcwd()
# save_dir = os.path.join(BASE_DIR, '..', 'training_results', 'cmae', 'cifar10', f'{len(selected_classes)}_classes')

save_dir = os.path.join(BASE_DIR, '..', 'training_results', 'cmae', 'cifar100', f'{len(selected_classes)}_classes')
print(save_dir)

# writer_dir = os.path.join(BASE_DIR, '..', 'training_results', 'tb_cmae', 'cifar10', f'{len(selected_classes)}_classes')
writer_dir = os.path.join(BASE_DIR, '..', 'training_results', 'tb_cmae', 'cifar100',   f'{len(selected_classes)}_classes')

os.makedirs(save_dir, exist_ok=True)
os.makedirs(writer_dir, exist_ok=True)


timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
writer = SummaryWriter(log_dir=os.path.join(writer_dir, f'cmae_trening_1_{timestamp}'))

num_epochs = 250


history = {
    'train_loss': [], 'train_rec': [], 'train_con': [],
    'val_loss': [], 'val_rec': [], 'val_con': []
}

epoch_number = 0
best_val_loss = float('inf')
# Early stopping
patience = 50
epochs_no_improve = 0
early_stop = False
print("Starting training...")

for epoch in range(num_epochs):

    if early_stop:
        print(f"\nEarly stopping triggered after {epoch} epochs (no improvement for {patience} epochs)")
        break

    model.train()
    train_loss = 0.0
    train_rec_loss = 0.0
    train_con_loss = 0.0 

    for batch_idx, (image, _) in enumerate(train_loader):
        image = image.to(device)

        outputs = model(image)
        # loss - cmae daje 3 wartosci
        loss, rec_loss, con_loss = model.compute_loss(outputs)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


        train_loss += loss.item()
        train_rec_loss += rec_loss.item()
        train_con_loss += con_loss.item()

        if batch_idx % 100 == 0:
            avg_loss = train_loss / (batch_idx + 1)
            avg_rec = train_rec_loss / (batch_idx + 1)
            avg_con = train_con_loss / (batch_idx + 1)

            print(f"  [{epoch+1}/{num_epochs}] Batch {batch_idx}/{len(train_loader)} "
                  f"Loss: {avg_loss:.4f} (REC: {avg_rec:.4f}, CON: {avg_con:.4f})")
    # update nauczyciela

    model.update_target()

    avg_train_loss = train_loss / len(train_loader)
    avg_train_rec = train_rec_loss / len(train_loader)
    avg_train_con = train_con_loss / len(train_loader)

    history['train_loss'].append(avg_train_loss)
    history['train_rec'].append(avg_train_rec)
    history['train_con'].append(avg_train_con)

    writer.add_scalar('Loss/Train', avg_train_loss, epoch_number)
    writer.add_scalar('Reconstruction_Loss/Train', avg_train_rec, epoch_number)
    writer.add_scalar('Contrastive_Loss/Train', avg_train_con, epoch_number)
    epoch_number += 1


    # validation
    model.eval()
    val_loss = 0.0
    val_rec_loss = 0.0
    val_con_loss = 0.0
    all_outputs = []

    with torch.no_grad():
        for image, _ in val_loader:
            image = image.to(device)

            outputs = model(image)
            loss, rec_loss, con_loss = model.compute_loss(outputs)

            val_loss += loss.item()
            val_rec_loss += rec_loss.item()
            val_con_loss += con_loss.item()

            all_outputs.append(outputs)

    if epoch % 5 == 0:
        outputs = all_outputs[0] 
        # wyciąganie danych do wizualizacji
        reconstructed = outputs['reconstructed_image']

        x_original, _, mask = outputs['loss_recon'] # oryginał i maska

        n_images = min(8, image.size(0))

        writer.add_images('Original', x_original[:n_images], epoch)
        writer.add_images('Reconstructed', reconstructed[:n_images], epoch)

        # wizualizacja maski
        mask_vis = mask[:n_images].repeat(1, 3, 1, 1)  # powielenuie kanały do 3
        writer.add_images('Mask', mask_vis, epoch)

        # wizualizacja zakodowanych reprezentacji -- tak widzial student
        masked_input = x_original[:n_images] * mask[:n_images]
        writer.add_images('Masked_Input', masked_input, epoch)

    avg_val_loss = val_loss / len(val_loader)
    avg_val_rec = val_rec_loss / len(val_loader)
    avg_val_con = val_con_loss / len(val_loader)

    history['val_loss'].append(avg_val_loss)
    history['val_rec'].append(avg_val_rec)
    history['val_con'].append(avg_val_con)

    writer.add_scalar('Loss/val_total', avg_val_loss, epoch)
    writer.add_scalar('Loss/val_rec', avg_val_rec, epoch)
    writer.add_scalar('Loss/val_con', avg_val_con, epoch)

    print(f"Epoch [{epoch+1}/{num_epochs}] ")
    print(f"  Train Loss: {avg_train_loss:.4f} (REC: {avg_train_rec:.4f}, CON: {avg_train_con:.4f})")
    print(f"  Val   Loss: {avg_val_loss:.4f} (REC: {avg_val_rec:.4f}, CON: {avg_val_con:.4f})")

    # zapisywanie najlepszego modelu
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        epochs_no_improve = 0
        checkpoint ={

            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': avg_train_loss,
            'val_loss': avg_val_loss,
            'latent_dim': 256,
            'train_losses': history['train_loss'],
            'val_losses': history['val_loss'],
            'selected_classes': selected_classes,
            
        }
        timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
        # checkpoint_path = os.path.join(save_dir, f'cmae_cifar10_best_{timestamp}.pt')
        checkpoint_path = os.path.join(save_dir, f'cmae_cifar100_best_trening_1_{timestamp}.pt')
        torch.save(checkpoint, checkpoint_path)
        # print(f" New best model saved! (Val loss improved)")
    else:
        epochs_no_improve += 1
        # print(f"No improvement for {epochs_no_improve} epoch(s)")

        if epochs_no_improve >= patience:
            early_stop = True

# zapisywanie historii treningu
df = pd.DataFrame({
    'epoch': range(1, len(history['train_loss']) + 1),
    'train_loss': history['train_loss'],
    'val_loss': history['val_loss'],
    'train_rec': history['train_rec'],
    'val_rec': history['val_rec'],
    'train_con': history['train_con'],
    'val_con': history['val_con']


})

timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
# history_csv = os.path.join(save_dir, f'cmae_cifar10_training_results_{timestamp}.csv')
history_csv = os.path.join(save_dir, f'cmae_cifar100_training_results_1_{timestamp}.csv')
df.to_csv(history_csv, index=False)

C:\Users\aczar\Desktop\polibuda\ML_projekt\src\notebooks_test_train\..\training_results\cmae\cifar100\5_classes
Starting training...
  [1/250] Batch 0/32 Loss: 6.0647 (REC: 1.1139, CON: 4.9509)
Epoch [1/250] 
  Train Loss: 3.7082 (REC: 0.8638, CON: 2.8445)
  Val   Loss: 5.4888 (REC: 1.0022, CON: 4.4866)
  [2/250] Batch 0/32 Loss: 3.1639 (REC: 0.8107, CON: 2.3532)
Epoch [2/250] 
  Train Loss: 2.9427 (REC: 0.8237, CON: 2.1190)
  Val   Loss: 3.7270 (REC: 0.8175, CON: 2.9096)
  [3/250] Batch 0/32 Loss: 2.9819 (REC: 0.8292, CON: 2.1527)
Epoch [3/250] 
  Train Loss: 2.5926 (REC: 0.8194, CON: 1.7732)
  Val   Loss: 3.2924 (REC: 0.8436, CON: 2.4488)
  [4/250] Batch 0/32 Loss: 2.5158 (REC: 0.7691, CON: 1.7466)
Epoch [4/250] 
  Train Loss: 2.4147 (REC: 0.8132, CON: 1.6015)
  Val   Loss: 3.0717 (REC: 0.8167, CON: 2.2550)
  [5/250] Batch 0/32 Loss: 2.2749 (REC: 0.7969, CON: 1.4781)
Epoch [5/250] 
  Train Loss: 2.2099 (REC: 0.8033, CON: 1.4066)
  Val   Loss: 2.8233 (REC: 0.8104, CON: 2.0128)
  [6/25